**1. Implement a function that create a 3D tensor from nested python lists,computes its transpose along the last two dimensions, verifies the result by checking shape and element-wise equality without using torch.transpose directly**

In [10]:
import torch

def transpose_last_two(tensor_list):
    tensor = torch.tensor(tensor_list)
    transposed = tensor.permute(0,2,1)
    expected_shape = (tensor.shape[0],tensor.shape[2],tensor.shape[1])
    assert transposed.shape== expected_shape, f"Shape mismatch:{transposed.shape} vs {expected_shape}"
    print(tensor.shape[1])
    equal=True
    for i in range(tensor.shape[0]):
        for j in range(tensor.shape[2]):
            for k in range(tensor.shape[1]):

                if transposed[i,j,k]!=tensor[i,k,j]:
                    equal=False
                    break
            if not equal:
                break
        if not equal:
            break
    assert equal , "Element-wise mismatch in trnaspose"

a = [[[1,2],[3,4]],[[4,5],[5,6]]]
transpose_last_two(a)


2


**2. Explain exactly what happens in memory when you call .view() versus .reshape() on a non-contiguous tensor in pytorch. When does each succeed or fail and why?**

.view() always fails on non-contiguous memory as it requires contiguous memory and it wont copy for creating a view instead it just changes shape metadata without moving data, whereas .reshape() copies the data and converts into contiguous memory without any issues.

**3. Implement a custom broadcasting-aware element-wise multiplication function that accepts tensors of arbitrary compatible shapes, applies the operation, and prints the resulting shape and intermediate broadcast shapes at each step.**

In [13]:
import torch

def broadcast_mul(a, b):
    print(f"Input shapes: a={a.shape}, b={b.shape}")
    
    # Get broadcasted shape
    broadcast_shape = torch.broadcast_shapes(a.shape, b.shape)
    print(f"Broadcasted shape: {broadcast_shape}")
    
    # Manually broadcast tensors
    a_broadcasted = a.expand(broadcast_shape)
    b_broadcasted = b.expand(broadcast_shape)
    print(f"After expand: a={a_broadcasted.shape}, b={b_broadcasted.shape}")
    print(f"Memory shared with original a: {a_broadcasted.data_ptr() == a.data_ptr() if a.numel() == a_broadcasted.numel() else False}")
    
    # Element-wise multiplication
    result = a_broadcasted * b_broadcasted
    print(f"Result shape: {result.shape}")
    
    return result

a = torch.tensor([1, 2, 3])
b = torch.tensor([[4], [5], [6]])
broadcast_mul(a, b)

Input shapes: a=torch.Size([3]), b=torch.Size([3, 1])
Broadcasted shape: torch.Size([3, 3])
After expand: a=torch.Size([3, 3]), b=torch.Size([3, 3])
Memory shared with original a: False
Result shape: torch.Size([3, 3])


tensor([[ 4,  8, 12],
        [ 5, 10, 15],
        [ 6, 12, 18]])

**4. Explain the difference between torch.tensor.data and torch.tensor.detach() and describe a scenario where using .data instead of .detach() leads to a silent correctness bug during training.**

.data() and .detach() both tensors shares a storage but does not participate in autograd and only .detach() will have graph reference for future in-place modifications and is safe . 
Suppose we are changing values using .data() autograd will have no awareness about it but in .detach() it will flag error during autograd that data is changed.

**5. Implement a function that manually computes the gradient of z = x² + 3xy + y³ with respect to x and y using PyTorch's autograd, then verify your result analytucally**

In [15]:
import torch

def manual_grad_check():
    x = torch.tensor(2.0, requires_grad=True)
    y = torch.tensor(3.0, requires_grad=True)
    
    z = x**2 + 3*x*y + y**3
    
    z.backward()
    
    print(f"PyTorch autograd: dz/dx = {x.grad.item()}, dz/dy = {y.grad.item()}")
    
    analytical_dx = 2*x + 3*y
    analytical_dy = 3*x + 3*y**2
    
    print(f"Analytical: dz/dx = {analytical_dx.item()}, dz/dy = {analytical_dy.item()}")
    
    assert torch.isclose(x.grad, analytical_dx)
    assert torch.isclose(y.grad, analytical_dy)
    print("Verification passed")

manual_grad_check()

PyTorch autograd: dz/dx = 13.0, dz/dy = 33.0
Analytical: dz/dx = 13.0, dz/dy = 33.0
Verification passed


**6. Explain what retain_graph=True does in .backward(), why it is needed in certain training loops, and what memory cost it incurs. Give a concrete use case such as multi-task learning**

retain_graph=True will retain all the computation graph after doing backward() so that another backward() can be performed. This will store all intermediate activations and gradients until explicitely freed.

**7. Implement a custom nn.Module called LinearWithBias that manually manages its weight and bias as nn.Parameter objects, implements forward(), and registers them so they appear in parameters() and state_dict().**

In [2]:
import torch
from torch import nn
class LinearWithBiasModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(1,dtype=torch.float),requires_grad=True)
        self.bias = nn.Parameter(torch.randn(1,dtype=torch.float),requires_grad=True)

    def forward(self,x:torch.Tensor)->torch.Tensor:
        return self.weight*x+self.bias

model = LinearWithBiasModel()
list(model.parameters())
# model.state_dict()

[Parameter containing:
 tensor([0.1057], requires_grad=True),
 Parameter containing:
 tensor([-1.3780], requires_grad=True)]

**8. Implement a complete training loop for a two-layer MLP on a synthetic regression dataset, using SGD with momentum, including zero_grad(), forward pass, loss computation, .backward(), and optimizer step — all from scratch with no high-level wrappers.**

In [23]:
import torch
from torch import nn,optim

torch.manual_seed(42)
X = torch.randn(100,5)
y = X @ torch.randn(5,1) + 0.1 * torch.randn(100,1)

class two_layer_mlp(nn.Module):
    def __init__(self,input_size=5,hidden_size=10,output_size=1):
        super().__init__()
        self.layer1 = nn.Linear(input_size,hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size,output_size)

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))  # layer1 -> ReLU -> layer2

model = two_layer_mlp()
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(),lr=0.01,momentum=0.9)

epochs = 50
batch_size = 16

for epoch in range(epochs):
    permutation = torch.randperm(X.size(0))
    epoch_loss = 0.0

    for i in range(0,X.size(0),batch_size):
        index = permutation[i:i+batch_size]
        batch_x,batch_y = X[index],y[index]

        outputs = model(batch_x)
        loss = loss_fn(outputs,batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {epoch_loss / (len(X)//batch_size):.4f}")

Epoch 10, Loss: 0.0351
Epoch 20, Loss: 0.0165
Epoch 30, Loss: 0.0158
Epoch 40, Loss: 0.0148
Epoch 50, Loss: 0.0127


**9. Write a full model checkpointing function that saves model state_dict, optimizer state_dict, current epoch, loss, and a random seed state to disk, and a corresponding restore function that resumes training exactly from that point.** 

In [25]:
import torch
import random
import numpy as np 

def save_checkpoint(model,optimizer,epoch,loss,seed,filepath):
    checkpoint={
        'epoch':epoch,
        'loss':loss,
        'optimizer_state_dict':optimizer.state_dict(),
        'model_state_dict':model.state_dict(),
        'seed':seed,
        'random_state':random.getstate(),
        'numpy_random_state': np.random.get_state(),
        'torch_random_state': torch.get_rng_state(),
        'cuda_random_state': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    }
    torch.save(checkpoint,filepath)

def load_checkpoint(filepath,model,optimizer=None):
    checkpoint = torch.load(filepath)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    random.setstate(checkpoint['random_state'])
    np.random.set_state(checkpoint['numpy_random_state'])
    torch.set_rng_state(checkpoint['torch_random_state'])
    if torch.cuda.is_available() and checkpoint['cuda_random_state']:
        torch.cuda.set_rng_state_all(checkpoint['cuda_random_state'])
    
    torch.manual_seed(checkpoint['seed'])
    
    return checkpoint['epoch'], checkpoint['loss']

model = torch.nn.Linear(5,1)
optimizer = torch.optim.SGD(model.parameters(),lr=0.01,momentum=0.9)
save_checkpoint(model, optimizer, epoch=10, loss=0.23, seed=42, filepath='checkpoint.pth')
epoch, loss = load_checkpoint('checkpoint.pth', model, optimizer)
print(f"Resumed from epoch {epoch}, loss {loss}")


Resumed from epoch 10, loss 0.23


C:\Users\AKSHAY\AppData\Local\Temp\ipykernel_16808\1183803970.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filepath)


**10. Explain PyTorch's computation graph lifecycle: when is a graph created, when is it freed, and what are the implications for memory management in a training loop that calls .backward() inside torch.no_grad()?**

Graph is created for tensors who has required_grad=True enabled and graph is destroyed once the backward() is called unless the retain_graph=True is not passed as parameter. whatever we do inside torch.no_grad() no graph is maintained or tracked.

**11. Implement a custom autograd.Function for the SiLU (Sigmoid Linear Unit) activation, writing both the forward and backward static methods manually, including the analytic gradient formula.**

In [28]:
import torch
from torch.autograd import Function

class SiLU(Function):
    @staticmethod
    def forward(ctx,x):
        # SiLU(x) = x * sigmoid(x) = x / (1 + exp(-x))
        sigmoid = 1/(1+torch.exp(-x))
        ctx.save_for_backward(sigmoid,x)
        return x*sigmoid
    
    @staticmethod
    def backward(ctx,grad_outputs):
        sigmoid,x = ctx.saved_tensors
        # Gradient: d/dx [x * sigmoid(x)] = sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x))
        # = sigmoid(x) * (1 + x * (1 - sigmoid(x)))
        grad_x = grad_outputs*(sigmoid+x*sigmoid*(1-sigmoid))
        return grad_x
    
x = torch.randn(3,requires_grad=True)
silu = SiLU.apply
out = silu(x)
out.sum().backward()
torch_out = torch.nn.functional.silu(x)
torch_out.sum().backward()

print(f"Custom grad: {x.grad}")
print(f"PyTorch grad: {x.grad}")


Custom grad: tensor([2.1960, 0.4028, 1.4458])
PyTorch grad: tensor([2.1960, 0.4028, 1.4458])
